# Phase 4: RNN/LSTM Sequence Tracking for 3D Hand Gesture Recognition
## Tuần 5–6: Temporal Modeling with Long Short-Term Memory Networks

---

### Overview

In this notebook, we move from static, frame-level classification to **temporal sequence modeling**.
A hand gesture is not a single pose — it is a **trajectory through time**: a sequence of 15 frames,
each containing 21 hand landmarks in 3D space (63 features). To classify the gesture correctly,
the model must learn how the hand *moves*, not just where it *is*.

### Why LSTM?

Recurrent Neural Networks (RNNs) process sequences step-by-step, maintaining an internal
**hidden state** that accumulates information from all previous time steps. However, vanilla RNNs
suffer from **vanishing gradients**: as the sequence grows, gradients shrink exponentially during
backpropagation through time, making it impossible to learn long-range dependencies.

**Long Short-Term Memory (LSTM)** networks solve this with a gated architecture:

| Gate | Purpose |
|------|---------|  
| **Forget gate** $f_t$ | Decides what old information to discard from the cell state |
| **Input gate** $i_t$ | Decides what new information to store in the cell state |
| **Output gate** $o_t$ | Decides what part of the cell state to expose as the hidden state |

The **cell state** $C_t$ acts as an information highway that flows through the entire sequence
with minimal modification, enabling gradients to flow backward through many time steps.

### Architecture Design

```
Input (batch, 15, 63)
    │
    ▼
Feature Embedding: Linear(63 → 128) + LayerNorm + ReLU
    │
    ▼
LSTM: 2 layers, hidden_size=128, dropout=0.3
    │
    ▼
Final hidden state h_n[-1]: (batch, 128)
    │
    ▼
Classifier: Linear(128 → num_classes)
```

#### Why a Feature Embedding Layer?

The raw 63-dimensional input contains X, Y, Z coordinates of 21 landmarks. These coordinates
live in a relatively low-dimensional space with heterogeneous scales. The embedding layer:

1. **Projects** the raw coordinates into a richer 128-dimensional feature space
2. **Normalizes** features via LayerNorm, ensuring stable LSTM training
3. **Introduces non-linearity** via ReLU, allowing the network to learn useful representations
   *before* temporal processing begins

This reduces the LSTM's burden: instead of learning both spatial patterns and temporal dynamics
simultaneously, the embedding layer handles spatial feature extraction while the LSTM focuses
purely on temporal relationships.

#### What Does the Hidden State Represent?

After processing all 15 frames, the LSTM's final hidden state $h_{T}$ is a **128-dimensional
summary vector** that encodes the entire gesture trajectory. It captures:

- **Motion direction and speed** (how landmarks move between frames)
- **Temporal patterns** (acceleration, pauses, rhythmic movements)
- **Gesture-specific signatures** (unique trajectories that distinguish one gesture from another)

We will visualize these hidden states using t-SNE and PCA to verify that the LSTM learns
separable representations for different gesture classes.

## 1. Environment Setup and Imports

In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── Device configuration ─────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_PATH = r'D:\WORKSAPCE\Monitor_AR\1_AI_Model_Training\Dataset\raw_data.csv'
MODEL_DIR = r'D:\WORKSAPCE\Monitor_AR\1_AI_Model_Training\Models'
BEST_MODEL_PATH = os.path.join(MODEL_DIR, 'best_lstm.pth')

os.makedirs(MODEL_DIR, exist_ok=True)
print(f"Data path  : {DATA_PATH}")
print(f"Model save : {BEST_MODEL_PATH}")

## 2. Data Loading and Exploration

The dataset is stored in CSV format where each **row** represents a single frame of a gesture
sample. Each sample consists of 15 consecutive frames (`frame_index` 0–14), and each frame
captures 21 hand landmarks with 3D coordinates (X, Y, Z), giving us 63 features per frame.

We need to:
1. Load the CSV and verify its structure
2. Identify the feature columns (landmark coordinates)
3. Examine the class distribution
4. Reshape the data from per-frame rows into per-sample sequences of shape (15, 63)

In [ ]:
# ── Load the raw data ────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
print(f"DataFrame shape: {df.shape}")
print(f"Columns ({len(df.columns)}): {list(df.columns[:10])} ... {list(df.columns[-5:])}")
print()
df.head()

In [ ]:
# ── Identify landmark feature columns ────────────────────────────────────────
meta_cols = ['sample_id', 'frame_index', 'label']
feature_cols = [c for c in df.columns if c not in meta_cols]
print(f"Number of feature columns: {len(feature_cols)}")
print(f"Feature columns: {feature_cols[:9]} ... {feature_cols[-3:]}")
assert len(feature_cols) == 63, f"Expected 63 feature columns, got {len(feature_cols)}"

# ── Basic statistics ──────────────────────────────────────────────────────────
n_samples = df['sample_id'].nunique()
n_frames_per_sample = df.groupby('sample_id')['frame_index'].nunique()
n_classes = df['label'].nunique()
class_names = sorted(df['label'].unique(), key=str)

print(f"\nTotal unique samples : {n_samples}")
print(f"Frames per sample    : min={n_frames_per_sample.min()}, max={n_frames_per_sample.max()}")
print(f"Number of classes    : {n_classes}")
print(f"Class names          : {class_names}")

In [ ]:
# ── Class distribution ────────────────────────────────────────────────────────
class_counts = df.groupby('sample_id')['label'].first().value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 5))
palette = sns.color_palette('Set2', n_colors=len(class_counts))
bars = ax.bar(range(len(class_counts)), class_counts.values, color=palette, edgecolor='white', linewidth=0.8)
ax.set_xticks(range(len(class_counts)))
ax.set_xticklabels(class_counts.index, rotation=45, ha='right', fontsize=10)
ax.set_xlabel('Gesture Class', fontsize=12)
ax.set_ylabel('Number of Samples', fontsize=12)
ax.set_title('Class Distribution in Dataset', fontsize=14, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)

# Add count labels on bars
for bar, count in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            str(count), ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Data Preprocessing Pipeline

### Preprocessing Steps

1. **Label Encoding**: Convert string labels to integer class indices using `LabelEncoder`
2. **Sequence Assembly**: Group the per-frame rows by `sample_id` and stack them into
   (15, 63) arrays — one 15-frame sequence per sample
3. **StandardScaler Normalization**: Fit the scaler on the training set and transform all splits.
   We flatten each sequence to (15×63,) for scaling, then reshape back to (15, 63)
4. **Train/Val/Test Split**: 70% train, 15% validation, 15% test (stratified by label)

In [ ]:
# ── Label encoding ────────────────────────────────────────────────────────────
label_encoder = LabelEncoder()
df['label_encoded'] = label_encoder.fit_transform(df['label'])

num_classes = len(label_encoder.classes_)
print(f"Label mapping ({num_classes} classes):")
for idx, name in enumerate(label_encoder.classes_):
    print(f"  {idx}: {name}")

In [ ]:
# ── Assemble sequences: group by sample_id → (15, 63) arrays ────────────────
sequences = []
labels = []

for sample_id, group in df.groupby('sample_id'):
    # Sort by frame_index to ensure temporal order
    group_sorted = group.sort_values('frame_index')
    seq = group_sorted[feature_cols].values  # shape: (15, 63)
    lbl = group_sorted['label_encoded'].iloc[0]
    
    if seq.shape[0] == 15:  # Only keep complete sequences
        sequences.append(seq)
        labels.append(lbl)

X = np.array(sequences, dtype=np.float32)  # (N, 15, 63)
y = np.array(labels, dtype=np.int64)        # (N,)

print(f"Assembled sequences: X.shape = {X.shape}, y.shape = {y.shape}")
print(f"Data type: X={X.dtype}, y={y.dtype}")

In [ ]:
# ── Train / Validation / Test split (70 / 15 / 15) ─────────────────────────
# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
# Second split: 50% of temp = 15% val, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f"Train : {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Val   : {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test  : {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")

In [ ]:
# ── StandardScaler normalization ──────────────────────────────────────────────
# Flatten (N, 15, 63) → (N, 945) for scaling, then reshape back
n_train, seq_len, n_features = X_train.shape

scaler = StandardScaler()
X_train_flat = X_train.reshape(n_train, -1)
X_val_flat = X_val.reshape(X_val.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

# Fit on training data only
X_train_flat = scaler.fit_transform(X_train_flat)
X_val_flat = scaler.transform(X_val_flat)
X_test_flat = scaler.transform(X_test_flat)

# Reshape back to (N, 15, 63)
X_train = X_train_flat.reshape(-1, seq_len, n_features).astype(np.float32)
X_val = X_val_flat.reshape(-1, seq_len, n_features).astype(np.float32)
X_test = X_test_flat.reshape(-1, seq_len, n_features).astype(np.float32)

print(f"After scaling — X_train: mean={X_train.mean():.4f}, std={X_train.std():.4f}")
print(f"                X_val  : mean={X_val.mean():.4f}, std={X_val.std():.4f}")
print(f"                X_test : mean={X_test.mean():.4f}, std={X_test.std():.4f}")

## 4. PyTorch Dataset and DataLoader

We create a custom `Dataset` that returns:
- **Sequence tensor**: shape (15, 63) — the full 15-frame gesture trajectory
- **Label tensor**: scalar integer — the gesture class index

Unlike convolutional approaches that might process frames independently, the LSTM receives
the **entire sequence** at once and processes it step-by-step internally.

In [ ]:
class GestureSequenceDataset(Dataset):
    """Dataset for 3D hand gesture sequences.
    
    Each sample is a full 15-frame sequence of 63-dimensional landmark features.
    Returns:
        sequence (Tensor): shape (15, 63) — temporal sequence of hand landmarks
        label (Tensor): scalar — gesture class index
    """
    
    def __init__(self, sequences, labels):
        self.sequences = torch.tensor(sequences, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]


# ── Create datasets ──────────────────────────────────────────────────────────
train_dataset = GestureSequenceDataset(X_train, y_train)
val_dataset = GestureSequenceDataset(X_val, y_val)
test_dataset = GestureSequenceDataset(X_test, y_test)

# ── Create dataloaders ────────────────────────────────────────────────────────
BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Verify shapes
sample_seq, sample_lbl = next(iter(train_loader))
print(f"Batch sequence shape : {sample_seq.shape}  (batch, seq_len, features)")
print(f"Batch label shape    : {sample_lbl.shape}")
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

## 5. LSTM Model Architecture

### Architecture Details

The model has three main components:

1. **Feature Embedding** (`nn.Linear(63, 128) + LayerNorm + ReLU`):
   - Applied independently to each frame in the sequence
   - Projects raw 63D coordinates → 128D learned feature space
   - LayerNorm stabilizes training by normalizing across the feature dimension
   - ReLU introduces non-linearity for richer representations

2. **LSTM** (`nn.LSTM(128, 128, num_layers=2, dropout=0.3)`):
   - 2 stacked LSTM layers for hierarchical temporal feature learning
   - Layer 1 learns low-level motion patterns (velocity, direction)
   - Layer 2 learns high-level gesture semantics (gesture type, phase)
   - Dropout between layers prevents co-adaptation of temporal features
   - `batch_first=True` for convenient (batch, seq, features) format

3. **Classifier** (`nn.Linear(128, num_classes)`):
   - Takes the final hidden state from the last LSTM layer: `h_n[-1]`
   - This single 128D vector summarizes the entire 15-frame trajectory
   - Maps it to class logits for CrossEntropy loss

### Why Use `h_n[-1]` Instead of All Outputs?

The LSTM produces an output at every time step, but for sequence classification,
the **final hidden state** is often the best choice because:
- It has "seen" the entire sequence
- It serves as a compressed summary vector
- It avoids the need for an additional aggregation mechanism (attention, pooling)

In [ ]:
class LSTMGestureClassifier(nn.Module):
    """LSTM-based model for 3D hand gesture sequence classification.
    
    Architecture:
        Input (batch, 15, 63)
          → Feature Embedding: Linear(63, 128) + LayerNorm + ReLU
          → LSTM(128, 128, num_layers=2, dropout=0.3)
          → Final hidden state h_n[-1]: (batch, 128)
          → Classifier: Linear(128, num_classes)
    """
    
    def __init__(self, input_size=63, embed_size=128, hidden_size=128,
                 num_layers=2, num_classes=10, dropout=0.3):
        super(LSTMGestureClassifier, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # ── Feature Embedding Layer ──────────────────────────────────────────
        self.embedding = nn.Sequential(
            nn.Linear(input_size, embed_size),
            nn.LayerNorm(embed_size),
            nn.ReLU()
        )
        
        # ── LSTM Layers ──────────────────────────────────────────────────────
        self.lstm = nn.LSTM(
            input_size=embed_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        
        # ── Classifier Head ──────────────────────────────────────────────────
        self.classifier = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x, return_hidden=False):
        """
        Args:
            x: Input tensor of shape (batch, seq_len, 63)
            return_hidden: If True, also return the final hidden state
                           for visualization purposes
        Returns:
            logits: (batch, num_classes)
            h_final: (batch, hidden_size) — only if return_hidden=True
        """
        # Apply embedding to each frame independently
        # x: (batch, 15, 63) → (batch, 15, 128)
        x = self.embedding(x)
        
        # LSTM forward pass
        # output: (batch, 15, 128) — hidden state at each time step
        # h_n: (num_layers, batch, 128) — final hidden state per layer
        # c_n: (num_layers, batch, 128) — final cell state per layer
        output, (h_n, c_n) = self.lstm(x)
        
        # Use the final hidden state from the last layer
        # h_n[-1]: (batch, 128)
        h_final = h_n[-1]
        
        # Classify
        logits = self.classifier(h_final)
        
        if return_hidden:
            return logits, h_final
        return logits


# ── Instantiate model ─────────────────────────────────────────────────────────
model = LSTMGestureClassifier(
    input_size=63,
    embed_size=128,
    hidden_size=128,
    num_layers=2,
    num_classes=num_classes,
    dropout=0.3
).to(device)

print(model)
print(f"\n{'='*60}")

# ── Count parameters ──────────────────────────────────────────────────────────
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")

# ── Verify forward pass ───────────────────────────────────────────────────────
with torch.no_grad():
    dummy_input = torch.randn(4, 15, 63).to(device)
    logits, hidden = model(dummy_input, return_hidden=True)
    print(f"\nForward pass test:")
    print(f"  Input shape  : {dummy_input.shape}")
    print(f"  Logits shape : {logits.shape}")
    print(f"  Hidden shape : {hidden.shape}")

## 6. Training Configuration

### Optimizer and Loss

- **Adam optimizer** with learning rate 1e-3 and weight decay 1e-4 (L2 regularization)
- **CrossEntropyLoss** combines LogSoftmax and NLLLoss — standard for multi-class classification

### Early Stopping Strategy

We monitor **validation loss** with a patience of 15 epochs:
- If validation loss doesn't improve for 15 consecutive epochs, training stops
- The best model weights (lowest validation loss) are saved and restored
- This prevents overfitting while ensuring we capture the optimal model state

In [ ]:
class EarlyStopping:
    """Early stopping to terminate training when validation loss stops improving.
    
    Args:
        patience: Number of epochs to wait for improvement
        min_delta: Minimum change in monitored value to qualify as improvement
        path: Path to save the best model checkpoint
    """
    
    def __init__(self, patience=15, min_delta=0.0, path='best_model.pth'):
        self.patience = patience
        self.min_delta = min_delta
        self.path = path
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.best_epoch = 0
    
    def __call__(self, val_loss, model, epoch):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_epoch = epoch
            self._save_checkpoint(model)
        elif val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.best_epoch = epoch
            self.counter = 0
            self._save_checkpoint(model)
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
    
    def _save_checkpoint(self, model):
        torch.save(model.state_dict(), self.path)


# ── Training hyperparameters ──────────────────────────────────────────────────
MAX_EPOCHS = 150
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 15

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
early_stopping = EarlyStopping(patience=PATIENCE, path=BEST_MODEL_PATH)

print(f"Optimizer     : Adam (lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})")
print(f"Loss function : CrossEntropyLoss")
print(f"Max epochs    : {MAX_EPOCHS}")
print(f"Early stopping: patience={PATIENCE}, monitor=val_loss")
print(f"Batch size    : {BATCH_SIZE}")

## 7. Training Loop

The training loop processes each batch through the LSTM model:

1. **Forward pass**: The embedding layer processes each frame, then the LSTM reads the
   15-frame sequence step-by-step, updating its hidden state at each step
2. **Loss computation**: CrossEntropyLoss on the logits from the final hidden state
3. **Backward pass**: Gradients flow back through time (BPTT) to all 15 time steps
4. **Parameter update**: Adam optimizer adjusts all weights

We track both loss and accuracy on the training and validation sets at every epoch.

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch and return average loss and accuracy."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for sequences, labels in dataloader:
        sequences = sequences.to(device)  # (batch, 15, 63)
        labels = labels.to(device)         # (batch,)
        
        # Forward pass
        logits = model(sequences)           # (batch, num_classes)
        loss = criterion(logits, labels)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        # Accumulate statistics
        running_loss += loss.item() * sequences.size(0)
        _, predicted = torch.max(logits, dim=1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


def evaluate(model, dataloader, criterion, device):
    """Evaluate model on a dataset and return average loss and accuracy."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for sequences, labels in dataloader:
            sequences = sequences.to(device)
            labels = labels.to(device)
            
            logits = model(sequences)
            loss = criterion(logits, labels)
            
            running_loss += loss.item() * sequences.size(0)
            _, predicted = torch.max(logits, dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [ ]:
# ── Training loop ────────────────────────────────────────────────────────────
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}

print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>10} | {'Val Acc':>9} | {'ES':>3}")
print('-' * 65)

for epoch in range(1, MAX_EPOCHS + 1):
    # Train
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    
    # Record history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Early stopping check
    early_stopping(val_loss, model, epoch)
    
    # Print progress every 5 epochs or on significant events
    if epoch % 5 == 0 or epoch == 1 or early_stopping.early_stop:
        es_indicator = f"{early_stopping.counter}/{PATIENCE}"
        print(f"{epoch:>6} | {train_loss:>10.4f} | {train_acc:>8.4f} | {val_loss:>10.4f} | {val_acc:>8.4f} | {es_indicator:>3}")
    
    if early_stopping.early_stop:
        print(f"\n{'='*65}")
        print(f"Early stopping triggered at epoch {epoch}!")
        print(f"Best model saved at epoch {early_stopping.best_epoch} "
              f"with val_loss = {early_stopping.best_loss:.4f}")
        break

if not early_stopping.early_stop:
    print(f"\nTraining completed for all {MAX_EPOCHS} epochs.")
    print(f"Best model at epoch {early_stopping.best_epoch} "
          f"with val_loss = {early_stopping.best_loss:.4f}")

# ── Load best model weights ──────────────────────────────────────────────────
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
print(f"\nBest model weights loaded from: {BEST_MODEL_PATH}")

## 8. Training History Visualization

We plot the training and validation curves to analyze:
- **Loss curves**: Convergence behavior, overfitting detection
- **Accuracy curves**: Learning progress and generalization gap

The vertical dashed line marks the epoch where the best model was saved.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

epochs_range = range(1, len(history['train_loss']) + 1)
best_epoch = early_stopping.best_epoch

# ── Color palette ─────────────────────────────────────────────────────────────
c_train = '#2196F3'   # Blue
c_val = '#FF5722'     # Deep Orange
c_best = '#4CAF50'    # Green

# ── Loss curves ──────────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(epochs_range, history['train_loss'], color=c_train, linewidth=2, label='Train Loss', alpha=0.9)
ax.plot(epochs_range, history['val_loss'], color=c_val, linewidth=2, label='Val Loss', alpha=0.9)
ax.axvline(x=best_epoch, color=c_best, linestyle='--', linewidth=1.5, alpha=0.7, label=f'Best Epoch ({best_epoch})')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)

# ── Accuracy curves ──────────────────────────────────────────────────────────
ax = axes[1]
ax.plot(epochs_range, [a * 100 for a in history['train_acc']], color=c_train, linewidth=2, label='Train Acc', alpha=0.9)
ax.plot(epochs_range, [a * 100 for a in history['val_acc']], color=c_val, linewidth=2, label='Val Acc', alpha=0.9)
ax.axvline(x=best_epoch, color=c_best, linestyle='--', linewidth=1.5, alpha=0.7, label=f'Best Epoch ({best_epoch})')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Training & Validation Accuracy', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, framealpha=0.9, loc='lower right')
ax.grid(True, alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

# ── Print best metrics ────────────────────────────────────────────────────────
best_idx = best_epoch - 1
print(f"\nBest Model Metrics (Epoch {best_epoch}):")
print(f"  Train Loss: {history['train_loss'][best_idx]:.4f}  |  Train Acc: {history['train_acc'][best_idx]*100:.2f}%")
print(f"  Val Loss  : {history['val_loss'][best_idx]:.4f}  |  Val Acc  : {history['val_acc'][best_idx]*100:.2f}%")

## 9. Test Set Evaluation

We evaluate the best model on the held-out test set to get an unbiased estimate
of the model's generalization performance. We also generate:
- **Classification Report**: Precision, recall, and F1-score per class
- **Confusion Matrix**: To identify which gestures are frequently confused

In [ ]:
# ── Test evaluation ──────────────────────────────────────────────────────────
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"Test Loss    : {test_loss:.4f}")
print(f"Test Accuracy: {test_acc * 100:.2f}%")

# ── Get predictions for classification report ─────────────────────────────────
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for sequences, labels in test_loader:
        sequences = sequences.to(device)
        logits = model(sequences)
        _, predicted = torch.max(logits, dim=1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# ── Classification Report ────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("Classification Report:")
print("=" * 60)
target_names = [str(c) for c in label_encoder.classes_]
print(classification_report(all_labels, all_preds, target_names=target_names))

In [ ]:
# ── Confusion Matrix ─────────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names,
            linewidths=0.5, linecolor='white',
            cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('Confusion Matrix — LSTM Model (Test Set)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 10. Hidden State Visualization (t-SNE & PCA)

### What Are We Visualizing?

After the LSTM processes all 15 frames of a gesture, its final hidden state `h_n[-1]` is a
**128-dimensional vector** that encodes the entire gesture trajectory. If the LSTM has learned
meaningful temporal representations, gestures of the same class should cluster together in this
128D space.

Since we cannot directly visualize 128 dimensions, we use two dimensionality reduction techniques:

### t-SNE (t-distributed Stochastic Neighbor Embedding)
- **Non-linear** dimensionality reduction optimized for visualization
- Preserves **local structure**: nearby points in 128D remain nearby in 2D
- `perplexity=30` balances attention between local and global structure
- Best for revealing cluster structure and separation between classes

### PCA (Principal Component Analysis)
- **Linear** dimensionality reduction that finds directions of maximum variance
- Preserves **global structure**: the overall geometry of the data
- Faster than t-SNE and deterministic
- Good for understanding the dominant axes of variation

### Interpretation Guide
- **Well-separated clusters** → LSTM learned distinct temporal signatures per gesture
- **Overlapping clusters** → Those gestures have similar motion patterns
- **Compact clusters** → Low intra-class variance; the model is confident
- **Scattered points** → High variability within that gesture class

In [ ]:
# ── Extract hidden states from the entire test set ─────────────────────────
hidden_states = []
hidden_labels = []

model.eval()
with torch.no_grad():
    for sequences, labels in test_loader:
        sequences = sequences.to(device)
        _, h_final = model(sequences, return_hidden=True)
        hidden_states.append(h_final.cpu().numpy())
        hidden_labels.append(labels.numpy())

hidden_states = np.concatenate(hidden_states, axis=0)  # (N_test, 128)
hidden_labels = np.concatenate(hidden_labels, axis=0)  # (N_test,)

print(f"Hidden states shape: {hidden_states.shape}")
print(f"Hidden labels shape: {hidden_labels.shape}")
print(f"Unique classes: {np.unique(hidden_labels)}")

In [ ]:
# ── t-SNE Reduction ───────────────────────────────────────────────────────────
print("Running t-SNE (this may take a moment)...")
tsne = TSNE(
    n_components=2,
    perplexity=30,
    random_state=SEED,
    n_iter=1000,
    learning_rate='auto',
    init='pca'
)
hidden_tsne = tsne.fit_transform(hidden_states)
print(f"t-SNE output shape: {hidden_tsne.shape}")

# ── PCA Reduction ─────────────────────────────────────────────────────────────
print("Running PCA...")
pca = PCA(n_components=2, random_state=SEED)
hidden_pca = pca.fit_transform(hidden_states)
print(f"PCA output shape: {hidden_pca.shape}")
print(f"PCA explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {pca.explained_variance_ratio_.sum()*100:.1f}%")

In [ ]:
# ── Visualization: t-SNE and PCA side-by-side ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Use tab10 colormap for distinct class colors
cmap = plt.cm.get_cmap('tab10', num_classes)
class_labels_str = [str(c) for c in label_encoder.classes_]

# ── t-SNE plot ────────────────────────────────────────────────────────────────
ax = axes[0]
for class_idx in range(num_classes):
    mask = hidden_labels == class_idx
    ax.scatter(
        hidden_tsne[mask, 0], hidden_tsne[mask, 1],
        c=[cmap(class_idx)], label=class_labels_str[class_idx],
        s=40, alpha=0.7, edgecolors='white', linewidths=0.3
    )
ax.set_xlabel('t-SNE Dimension 1', fontsize=12)
ax.set_ylabel('t-SNE Dimension 2', fontsize=12)
ax.set_title('t-SNE of LSTM Hidden States (Test Set)', fontsize=14, fontweight='bold')
ax.legend(title='Gesture Class', fontsize=9, title_fontsize=10,
          bbox_to_anchor=(1.0, 1.0), loc='upper left', framealpha=0.9,
          markerscale=1.2)
ax.grid(True, alpha=0.2)
ax.spines[['top', 'right']].set_visible(False)

# ── PCA plot ──────────────────────────────────────────────────────────────────
ax = axes[1]
for class_idx in range(num_classes):
    mask = hidden_labels == class_idx
    ax.scatter(
        hidden_pca[mask, 0], hidden_pca[mask, 1],
        c=[cmap(class_idx)], label=class_labels_str[class_idx],
        s=40, alpha=0.7, edgecolors='white', linewidths=0.3
    )
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)', fontsize=12)
ax.set_title('PCA of LSTM Hidden States (Test Set)', fontsize=14, fontweight='bold')
ax.legend(title='Gesture Class', fontsize=9, title_fontsize=10,
          bbox_to_anchor=(1.0, 1.0), loc='upper left', framealpha=0.9,
          markerscale=1.2)
ax.grid(True, alpha=0.2)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ── Detailed t-SNE plot with larger markers and annotations ────────────────
fig, ax = plt.subplots(figsize=(12, 10))

# Use a more colorful palette for the detailed view
palette = sns.color_palette('Set2', n_colors=num_classes) if num_classes <= 8 else \
          sns.color_palette('tab10', n_colors=num_classes)

for class_idx in range(num_classes):
    mask = hidden_labels == class_idx
    points = hidden_tsne[mask]
    
    # Plot scatter points
    ax.scatter(
        points[:, 0], points[:, 1],
        c=[palette[class_idx]], label=f'{class_labels_str[class_idx]} (n={mask.sum()})',
        s=60, alpha=0.65, edgecolors='white', linewidths=0.5, zorder=2
    )
    
    # Add centroid marker
    centroid = points.mean(axis=0)
    ax.scatter(
        centroid[0], centroid[1],
        c=[palette[class_idx]], s=200, marker='*',
        edgecolors='black', linewidths=1.0, zorder=3
    )

ax.set_xlabel('t-SNE Dimension 1', fontsize=13)
ax.set_ylabel('t-SNE Dimension 2', fontsize=13)
ax.set_title('t-SNE Visualization of LSTM Hidden States\n'
             '(Stars = Class Centroids)', fontsize=15, fontweight='bold')
ax.legend(title='Gesture Class', fontsize=10, title_fontsize=11,
          loc='best', framealpha=0.9, markerscale=1.0)
ax.grid(True, alpha=0.15, linestyle='--')
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

## 11. Summary and Analysis

### Key Findings

| Aspect | Details |
|--------|---------|  
| **Architecture** | Feature Embedding (63→128) + 2-layer LSTM (128 hidden) + Classifier |
| **Input** | 15-frame sequences of 63-dim hand landmarks |
| **Hidden State** | 128-dim summary vector encoding entire gesture trajectory |
| **Regularization** | Dropout (0.3), Weight Decay (1e-4), Gradient Clipping (1.0) |
| **Early Stopping** | Patience=15 on validation loss |

### How the LSTM Tracks Temporal Trajectories

At each time step $t$ (frame 0 to 14), the LSTM updates its hidden state:

$$h_t = \text{LSTM}(x_t, h_{t-1}, c_{t-1})$$

The hidden state $h_t$ at time $t$ encodes information about all frames seen so far
($x_0, x_1, \ldots, x_t$). By the final frame, $h_{14}$ contains a compressed
representation of the **entire gesture trajectory**.

The forget gate selectively erases irrelevant past information (e.g., the exact position
at frame 0 may not matter for distinguishing a swipe from a circle). The input gate
selectively writes new information (e.g., a sudden change in direction is highly
informative). This selective memory makes LSTMs effective at capturing the temporal
patterns that define each gesture.

### Why the Embedding Layer Helps

Without the embedding layer, the LSTM must simultaneously learn:
1. How to interpret raw (X, Y, Z) coordinates
2. How landmarks relate to each other spatially
3. How these spatial patterns evolve over time

The embedding layer handles tasks 1 and 2 by projecting raw coordinates into a learned
feature space where spatial relationships are already encoded. The LSTM can then focus
purely on task 3 — modeling temporal dynamics. This separation of concerns typically
leads to faster convergence and better final accuracy.

In [ ]:
# ── Final summary ────────────────────────────────────────────────────────────
print("=" * 60)
print("LSTM Gesture Recognition — Final Summary")
print("=" * 60)
print(f"  Model            : LSTMGestureClassifier")
print(f"  Parameters       : {trainable_params:,}")
print(f"  Best Epoch       : {early_stopping.best_epoch}")
print(f"  Best Val Loss    : {early_stopping.best_loss:.4f}")
print(f"  Test Loss        : {test_loss:.4f}")
print(f"  Test Accuracy    : {test_acc * 100:.2f}%")
print(f"  Model saved to   : {BEST_MODEL_PATH}")
print("=" * 60)